# Harmony: subset, reproject and transform data in the cloud

[Harmony](https://harmony.earthdata.nasa.gov/) is NASA's asynchronous,
cloud-based transformation service. `ea.plugins.harmony` submits a
transformation request, runs it remotely, and returns the output files.

It is the reference **remote, resumable** plugin: `request()` returns
immediately with a `Job` you can inspect, pause, resume or cancel.

## Setup

```bash
pip install earthaccess earthaccess-plugin-harmony jupyterlab matplotlib
```

All plugins except SlideRule use your Earthdata Login.


In [ ]:
import earthaccess as ea

ea.login()  # strategy="interactive" if netrc is not set up
ea.list_plugins(details=True)

## Search

We use MUR Sea Surface Temperature (SST) over the Gulf of Mexico.

In [ ]:
gulf = (-98.0, 18.0, -80.0, 31.0)  # west, south, east, north

sst = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    bounding_box=gulf,
    temporal=("2023-06-01", "2023-06-08"),
    count=2,
)
len(sst)

## What can Harmony do with this collection?

`supports()` / `capabilities()` report the operations and output formats
Harmony advertises for the collection:

In [ ]:
ea.plugins.harmony.capabilities("MUR-JPL-L4-GLOB-v4.1")

## A basic request: subset to the Gulf

`request()` returns immediately. `status()`/`info()` report progress and
the order URL; `ea.open`/`ea.download` block until it finishes.

In [ ]:
job = ea.plugins.harmony.request(sst, bbox=gulf)
job

In [ ]:
job.status()
job.info()  # includes an order_url; never the request payload

In [ ]:
# Open the transformed results, or download them:
files = ea.open(job)
# ea.download(job, "./data")
files

In [ ]:
import xarray as xr

# Harmony assets are files; open them with xarray.
ds = xr.open_mfdataset(files, combine="nested", concat_dim="time")
ds["analysed_sst"].isel(time=0).plot(figsize=(10, 6), cmap="RdYlBu_r")

## Subset by a GeoJSON polygon and selected variables

For an irregular area, pass a GeoJSON file with `shapefile=`. The file's
extension sets the shape mime type (`.geojson` / `.json` →
`application/geo+json`); `wkt=` does the same for an inline geometry.

`variables=` keeps only the bands you name, so the response is smaller.

In [ ]:
import json

# An arbitrary polygon over the Gulf of Mexico.
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-95.0, 25.0],
                        [-85.0, 25.0],
                        [-85.0, 30.0],
                        [-95.0, 30.0],
                        [-95.0, 25.0],
                    ]
                ],
            },
        }
    ],
}

with open("gulf_aoi.geojson", "w") as f:
    json.dump(aoi, f)

In [ ]:
job = ea.plugins.harmony.request(
    sst,
    shapefile="gulf_aoi.geojson",  # spatial subset from GeoJSON
    variables=["analysed_sst"],  # only this variable
)
files = ea.open(job)
files

In [ ]:
import xarray as xr

ds = xr.open_mfdataset(files, combine="nested", concat_dim="time")
ds  # only the requested variable is present

## More operations

The options mirror `harmony-py`'s `Request`, so every Harmony operation is
available — subsetting (`bbox`, `temporal`, `variables`, `shapefile`),
reprojection (`crs`, `width`, `height`, `scale_extent`, `interpolation`),
aggregation (`average`, `concatenate`) and reformatting (`output_format`):

```python
import datetime as dt
from earthaccess.plugins.harmony import HarmonyPlugin

job = ea.plugins.harmony.request(
    sst,
    bbox=(-165, 52, -140, 77),
    temporal={"start": dt.datetime(2023, 6, 1), "stop": dt.datetime(2023, 6, 8)},
    variables=["analysed_sst"],
    crs="EPSG:3995",          # reproject to a polar stereographic grid
    width=512,
    height=512,
    output_format="image/tiff",
    max_results=10,
)
```

Not every service supports every operation — check `capabilities()` first.

In [ ]:
import datetime as dt

# subset variables and time (no reprojection)
job = ea.plugins.harmony.request(
    sst,
    variables=["analysed_sst"],
    temporal={"start": dt.datetime(2023, 6, 1), "stop": dt.datetime(2023, 6, 8)},
    skip_preview=True,
)
ea.open(job)

## Steer the order

Harmony jobs are remote and resumable. Large requests enter a preview state
(`JobStatus.PAUSED`); `skip_preview()` accepts them.

In [ ]:
print(job.resumable)  # True
print(job.capabilities)  # {'cancel', 'pause', 'resume'}
# job.pause(); job.resume(); job.skip_preview(); job.cancel()
# ea.plugins.harmony.get_job(job.id)   # resume from another process

## Which plugins accept a dataset?

`list_services()` asks every installed plugin (no need to name one):

In [ ]:
sst.list_services()

## Summary

- `ea.plugins.harmony.request(results, **options)` → `Job` (remote, resumable).
- Options mirror harmony-py: subset, reproject, average, concatenate, format.
- `ea.open(job)` / `ea.download(job, path)` consume the output assets.